# ONNX core
In this notebook, we use two ways to make ONNX files usable.
1. Change ONNX model input dimension: remove the batch and track dimension, and keep only the feature dimension, which the linear layers directly act on.
2. Save the matrix and bias as text files.

Files are saved in the `./onnx_files_narrow_but_deep_untrained/simple` folder:
- The files named as `simple_[submodule_name].onnx` are the modified ONNX files
- The `txt` file are named as `[submodule_name]-[MatMul/Bias]_[id]_[shape].txt`.
  For `MatMul` (matrix) the shape is `[in_features/row]x[out_features/col]`.

In [4]:
from pathlib import Path
import onnx
import numpy as np

In [5]:
def last_dim_only(onnx_model):
    """
    Get a onnx model and get its input dimension. 
    Say the input dimesion is (None, 50, 6) for
    (batch_size, num_tracks, num_features), a linear 
    model in fact only acts on the last dimension.

    In this function, we manually changed the input 
    dimension of the model to just the last dimension.

    I am not sure whether this is enough "fool" the 
    software that uses the model, but let us give it a try.
    """
    input_tensor = onnx_model.graph.input[0]

    # Get the last dimension value (num_features)
    last_dim = input_tensor.type.tensor_type.shape.dim[-1].dim_value
    if last_dim == 0:
        raise ValueError("The last dimension is not a fixed value!")
    
    input_tensor.type.tensor_type.shape.dim.clear()
    dim = input_tensor.type.tensor_type.shape.dim.add()
    dim.dim_value = last_dim


def parse_linear_layers(onnx_model_path, save_path):
    """
    As our models are all sequential linear layers, we can 
    save the matrix and bias as text files one by one.
    """

    print(f'\n{onnx_model_path.stem}')
    
    onnx_model = onnx.load(onnx_model_path)
    input_tensor = onnx_model.graph.input[0]
    
    # input dimension is the in_features of the linear layer
    in_features = input_tensor.type.tensor_type.shape.dim[-1].dim_value
    
    # Here we assume the biases and matrices are in order.
    # We can do so since they are created as sequential model.
    matrices = []
    biases = []
    for tensor in onnx_model.graph.initializer:
        array = np.frombuffer(tensor.raw_data, dtype=np.float16)
        
        name = tensor.name
        print(f'{name}\t', end='')
        
        if 'MatMul' in name:
            assert len(array) % in_features == 0
            out_features = len(array) // in_features
            print(f'({in_features}, {out_features})')
            array = array.reshape(out_features, in_features)
            # save_name = f"{onnx_model_path.stem}-MatMul_{i//2}_{in_features}x{out_features}.txt"
            matrices.append((array, in_features, out_features))
            in_features = out_features
            
        elif 'bias' in name:
            print(f'({len(array)}, )')
            biases.append((array, len(array)))

    assert len(matrices) == len(biases)

    for i, (matrix, bias) in enumerate(zip(matrices, biases)):
        array, in_features, out_features = matrix
        save_name = f"{onnx_model_path.stem}-MatMul_{i}_{in_features}x{out_features}.txt"
        np.savetxt(Path(save_path)/save_name, array)

        array, num_features = bias
        save_name = f"{onnx_model_path.stem}-Bias_{i}_{num_features}.txt"
        np.savetxt(Path(save_path)/save_name, array)

In [8]:
precision = 'bf16'
onnx_root = Path("onnx_files_narrow")
onnx_simple = onnx_root/f'simple_{precision}'

if not onnx_simple.exists():
    onnx_simple.mkdir()

onnx_model_paths = list((onnx_root/f'submodules_{precision}').glob('submodule_*.onnx'))

for onnx_model_path in onnx_model_paths:
    # save a onnx model with simplified input shape
    onnx_model = onnx.load(onnx_model_path)
    last_dim_only(onnx_model)
    stem = onnx_model_path.stem
    fname = onnx_simple/f"simple_{stem}.onnx"
    onnx.save(onnx_model, fname)

    # save the weight and bias of the linear layers
    parse_linear_layers(onnx_model_path, onnx_simple)


submodule_output
bias	(27, )
onnx::MatMul_6	(128, 27)

submodule_solvers-2
1.bias	(128, )
3.linear.bias	(128, )
4.linear.bias	(128, )
5.linear.bias	(128, )
onnx::MatMul_25	(256, 128)
onnx::MatMul_26	(128, 128)
onnx::MatMul_27	(128, 128)
onnx::MatMul_28	(128, 128)

submodule_solvers-1
1.bias	(128, )
3.linear.bias	(128, )
4.linear.bias	(128, )
5.linear.bias	(128, )
onnx::MatMul_25	(256, 128)
onnx::MatMul_26	(128, 128)
onnx::MatMul_27	(128, 128)
onnx::MatMul_28	(128, 128)

submodule_solvers-0
1.bias	(128, )
3.linear.bias	(128, )
4.linear.bias	(128, )
5.linear.bias	(128, )
onnx::MatMul_25	(256, 128)
onnx::MatMul_26	(128, 128)
onnx::MatMul_27	(128, 128)
onnx::MatMul_28	(128, 128)

submodule_embed
1.bias	(128, )
4.bias	(128, )
onnx::MatMul_13	(6, 128)
onnx::MatMul_14	(128, 128)
